In [1]:
%%capture

import warnings

warnings.filterwarnings("ignore")
import calitp_portfolio.magics

import folium
import geopandas as gpd
import gcsfs
import google.auth
import pandas as pd

from great_tables import GT
import gt_extras as gte

import _ct_report_utils as utils
from update_vars import DIGEST_DICT, PROCESSED_GCS, abbrev_month

credentials, _ = google.auth.default()

In [2]:

from dotenv import dotenv_values

In [3]:
# Comment out and leave this cell right below pandas
district = "AD 75"

In [4]:
def readable_district_name(district_abbrev: str):

    return (
        district_abbrev
        .replace('AD', 'Assembly District')
        .replace('SD', 'Senate District')
    )

district_full_name = readable_district_name(district)

In [5]:
# %%capture_parameters
# district 

In [6]:
district_full_name

'Assembly District 75'

In [17]:
crosswalk_url = (
    f"{PROCESSED_GCS}"
    f"{DIGEST_DICT.crosswalk_legislative}_2026_04.parquet"
)

filesystem = gcsfs.GCSFileSystem()

try:
    # First, try an exact match using the Parquet filter.
    crosswalk_legislative = pd.read_parquet(
        crosswalk_url,
        filters=[
            ("legislative_district", "==", district),
        ],
        filesystem=filesystem,
    )

    # If the exact-match filter returns no rows,
    # reread the needed data and use str.contains instead.
    if crosswalk_legislative.empty:
        crosswalk_legislative = pd.read_parquet(
            crosswalk_url,
            columns=[
                "legislative_district",
                "analysis_name",
            ],
            filesystem=filesystem,
        )

        crosswalk_legislative = crosswalk_legislative[
            crosswalk_legislative["legislative_district"]
            .astype("string")
            .str.contains(
                str(district),
                case=False,
                regex=False,
                na=False,
            )
        ]

    list_of_operators = (
        crosswalk_legislative["analysis_name"]
        .dropna()
        .unique()
        .tolist()
    )

except Exception as error:
    print(
        f"Unable to read the legislative crosswalk for "
        f"district {district}: {error}"
    )
    raise

In [29]:
list_of_operators

['North County Transit District']

In [37]:
f"{PROCESSED_GCS}{DIGEST_DICT.operator_summary}.parquet"

'gs://calitp-analytics-data/data-analyses/gtfs_digest/processed/fct_monthly_operator_summary.parquet'

,Date,Analysis Name,Caltrans District,VP Name,TU Name,N Trips,Day Type,Daily Trips,Ttl Service Hours,N Routes,...,N Stops,Daily Arrivals,VP Messages Per Minute,N VP Trips,Daily VP Trips,Pct VP Trips,TU Messages Per Minute,N TU Trips,Daily TU Trips,Pct TU Trips


In [39]:
operator_raw_df = pd.read_parquet("gs://calitp-analytics-data/data-analyses/gtfs_digest/raw/fct_monthly_operator_summary_2026_10.parquet",
    filesystem = gcsfs.GCSFileSystem(),
    )

In [41]:
operator_raw_df.loc[operator_raw_df.schedule_name.str.contains("North County")]

,month,year,month_first_day,schedule_name,schedule_base64_url,vp_name,vp_base64_url,tu_name,tu_base64_url,day_type,...,n_days_schedule_only,n_days_schedule_and_rt,n_days_schedule_and_vp_only,n_days_schedule_and_tu_only,prediction_error_sec_array,prediction_error_sec_percentile_array,scaled_prediction_error_sec_array,pos_prediction_error_sec_array,neg_prediction_error_sec_array,pos_prediction_error_sec_percentile_array
132,11,2026,2026-11-01,North County Schedule,None,None,None,None,None,Sunday,...,NaN,NaN,NaN,NaN,[],[],[],[],[],[]
179,10,2026,2026-10-01,North County Schedule,None,None,None,None,None,Weekday,...,NaN,NaN,NaN,NaN,[],[],[],[],[],[]
281,11,2026,2026-11-01,North County Schedule,None,None,None,None,None,Weekday,...,NaN,NaN,NaN,NaN,[],[],[],[],[],[]
294,11,2026,2026-11-01,North County Schedule,None,None,None,None,None,Saturday,...,NaN,NaN,NaN,NaN,[],[],[],[],[],[]
310,9,2026,2026-09-01,North County Schedule,None,None,None,None,None,Sunday,...,NaN,NaN,NaN,NaN,[],[],[],[],[],[]
367,9,2026,2026-09-01,North County Schedule,None,None,None,None,None,Weekday,...,NaN,NaN,NaN,NaN,[],[],[],[],[],[]
623,10,2026,2026-10-01,North County Schedule,None,None,None,None,None,Saturday,...,NaN,NaN,NaN,NaN,[],[],[],[],[],[]
682,10,2026,2026-10-01,North County Schedule,None,None,None,None,None,Sunday,...,NaN,NaN,NaN,NaN,[],[],[],[],[],[]
732,9,2026,2026-09-01,North County Schedule,None,None,None,None,None,Saturday,...,NaN,NaN,NaN,NaN,[],[],[],[],[],[]


In [30]:
try:
    operator_summary_url = f"{PROCESSED_GCS}{DIGEST_DICT.operator_summary}.parquet"

    operator_df = pd.read_parquet(
    operator_summary_url,
    filesystem = gcsfs.GCSFileSystem(),
    filters=[
        ("Day Type", "==", "Weekday"),
        ("Analysis Name", "in", list_of_operators),
    ],
    ).merge(
    crosswalk_legislative.rename(columns = {"analysis_name": "Analysis Name"}),
    on = "Analysis Name",
    how = "inner"
    ).reset_index(drop=True)
except:
    pass

In [19]:
try:
    fct_monthly_routes_url = f"{PROCESSED_GCS}{DIGEST_DICT.route_map}.parquet"

    fct_monthly_route_df = gpd.read_parquet(
        fct_monthly_routes_url,
        storage_options = {"token": credentials.token},
        filters=[[("Analysis Name", "in", list_of_operators)]]
    ).reset_index(drop=True).pipe(
        utils.prep_gdf
    )
except:
    pass

In [27]:
fct_monthly_route_df

,Analysis Name,Route Name,Geometry,Route Length Miles


In [20]:
district_gdf = utils.load_legislative_district(district)

# {district}

Related products: **[Caltrans district](https://analysis.dds.dot.ca.gov/district_digest/)** and **[transit operator](https://analysis.dds.dot.ca.gov/gtfs_digest/)** reports

In [21]:
try:
    operator_df2 = utils.create_summary_table(
        operator_df, district_col = "legislative_district"
    ).iloc[:, :2]
except:
    pass

In [22]:
try:
    display(
        GT(operator_df2)
        .tab_header(
            title = "GTFS Summary Stats",
            subtitle=f"District {district}"
        )
    )
except:
    pass

## GTFS Stats by Operator
### Routes within {district}

In [23]:
CARTO_TILE_URL = dotenv_values("../_env")["CARTO_TILE_URL"]

In [24]:
m = district_gdf.explore(
        color = "#3a194f",
        tiles = CARTO_TILE_URL,
         attr="Basemap",
        name = "Legislative District Boundary",
        style_kwds = {"weight": 1, "fillOpacity": 0},
        highlight_kwds = {"fillOpacity": 0.2},
        tooltip=False
    )
    
    #color_map = cm.linear.Spectral_11.scale()
fct_monthly_route_df.explore(
        "Route Name",
        m = m,
        name = "Transit Routes in Legislative District",
        categorical = True,
        cmap = "Spectral",
        legend = False
        #marker_kwds={"fill": True},
        #style_kwds={"opacity": 0.5, "fillOpacity": 0.3}
    )
    
folium.LayerControl().add_to(m)
m

AxisError: axis 1 is out of bounds for array of dimension 0

In [ ]:
try:
    gtfs_table = utils.create_operator_table(
        operator_df, district_col = "legislative_district"
    ).drop(columns = "legislative_district")
except:
    pass

In [ ]:
try:
    display((
    GT(gtfs_table.sort_values("Daily Trips", ascending=False))
    .fmt_integer(
        columns=[c for c in gtfs_table.columns if c not in ["Operator"]]
    ).tab_header(
        title="Daily Weekday GTFS Stats by Operator",
        subtitle=f"District {district}",
    ).cols_align(
        columns=[c for c in gtfs_table.columns if c != "Operator"],
        align="center",
    ).pipe(
        gte.gt_color_box, 
        columns=["Daily Trips", "# Routes", "# Shapes", "# Stops", 
                 "Daily Arrivals", "Arrivals per Stop"],
        palette="YlGnBu", 
    )
))
except:
    pass